# Activity 5.01: Clustering Sales Data Using K-Means
**Dataset:** Sales Transactions Dataset Weekly (UCI) - weekly sales of 800 products over 52 weeks.

**Goal:** group products with similar sales trends using k-means, then experiment with the number of clusters to find a good value.

*Citation: Tan, S., & San Lau, J. (2014). Time series clustering: A superior alternative for market basket analysis. DaEng-2013, pp. 241-248.*

## 1. Import libraries

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

## 2. Load and inspect the data
The code first looks for `Sales_Transactions_Dataset_Weekly.csv` in the notebook folder. If it isn't there, it downloads it from the UCI repository.

In [2]:
fname = "Sales_Transactions_Dataset_Weekly.csv"
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00396/Sales_Transactions_Dataset_Weekly.csv"

try:
    df = pd.read_csv(fname)
except FileNotFoundError:
    df = pd.read_csv(url)

print(df.shape)
df.head()

(811, 107)


,Product_Code,W0,W1,W2,W3,W4,W5,W6,W7,W8,...,Normalized 42,Normalized 43,Normalized 44,Normalized 45,Normalized 46,Normalized 47,Normalized 48,Normalized 49,Normalized 50,Normalized 51
0,P1,11,12,10,8,13,12,14,21,6,...,0.06,0.22,0.28,0.39,0.50,0.00,0.22,0.17,0.11,0.39
1,P2,7,6,3,2,7,1,6,3,3,...,0.20,0.40,0.50,0.10,0.10,0.40,0.50,0.10,0.60,0.00
2,P3,7,11,8,9,10,8,7,13,12,...,0.27,1.00,0.18,0.18,0.36,0.45,1.00,0.45,0.45,0.36
3,P4,12,8,13,5,9,6,9,13,13,...,0.41,0.47,0.06,0.12,0.24,0.35,0.71,0.35,0.29,0.35
4,P5,8,5,13,11,6,7,9,14,9,...,0.27,0.53,0.27,0.60,0.20,0.20,0.13,0.53,0.33,0.40


Columns: `Product_Code`, `W0`-`W51` (raw weekly sales), `MIN`, `MAX`, and `Normalized 0`-`Normalized 51` (weekly sales scaled per product).

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 811 entries, 0 to 810
Columns: 107 entries, Product_Code to Normalized 51
dtypes: float64(52), int64(54), object(1)
memory usage: 678.1+ KB


## 3. Drop the unnecessary columns
The first 55 columns (`Product_Code`, 52 `W` columns, `MIN`, `MAX`) are removed, leaving only the normalized weekly columns. Clustering on normalized values groups products by the *shape* of their sales trend rather than by volume.

In [4]:
df_norm = df.copy()
df_norm.drop(df_norm.columns[:55], axis=1, inplace=True)
print(df_norm.shape)
df_norm.head()

(811, 52)


,Normalized 0,Normalized 1,Normalized 2,Normalized 3,Normalized 4,Normalized 5,Normalized 6,Normalized 7,Normalized 8,Normalized 9,...,Normalized 42,Normalized 43,Normalized 44,Normalized 45,Normalized 46,Normalized 47,Normalized 48,Normalized 49,Normalized 50,Normalized 51
0,0.44,0.50,0.39,0.28,0.56,0.50,0.61,1.00,0.17,0.61,...,0.06,0.22,0.28,0.39,0.50,0.00,0.22,0.17,0.11,0.39
1,0.70,0.60,0.30,0.20,0.70,0.10,0.60,0.30,0.30,0.30,...,0.20,0.40,0.50,0.10,0.10,0.40,0.50,0.10,0.60,0.00
2,0.36,0.73,0.45,0.55,0.64,0.45,0.36,0.91,0.82,0.27,...,0.27,1.00,0.18,0.18,0.36,0.45,1.00,0.45,0.45,0.36
3,0.59,0.35,0.65,0.18,0.41,0.24,0.41,0.65,0.65,0.53,...,0.41,0.47,0.06,0.12,0.24,0.35,0.71,0.35,0.29,0.35
4,0.33,0.13,0.67,0.53,0.20,0.27,0.40,0.73,0.40,0.40,...,0.27,0.53,0.27,0.60,0.20,0.20,0.13,0.53,0.33,0.40


## 4. Create the k-means model (8 clusters, `random_state=8`)

In [5]:
kmeans = KMeans(n_clusters=8, random_state=8)
kmeans.fit(df_norm)

,n_clusters,8
,init,'k-means++'
,n_init,'auto'
,max_iter,300
,tol,0.0001
,verbose,0
,random_state,8
,copy_x,True
,algorithm,'lloyd'


## 5. Retrieve the cluster labels

In [6]:
labels = kmeans.labels_
labels[:20]

array([6, 6, 2, 6, 6, 1, 6, 6, 2, 2, 6, 2, 5, 2, 5, 5, 5, 5, 5, 5],
      dtype=int32)

## 6. Keep only the `W` columns and add the labels
We go back to the original `df` and keep just the raw weekly sales columns, then attach each product's cluster label.

In [7]:
w_cols = [c for c in df.columns if c.startswith("W") and c[1:].isdigit()]
df_w = df[w_cols].copy()
df_w["label"] = labels
df_w.head()

,W0,W1,W2,W3,W4,W5,W6,W7,W8,W9,...,W43,W44,W45,W46,W47,W48,W49,W50,W51,label
0,11,12,10,8,13,12,14,21,6,14,...,7,8,10,12,3,7,6,5,10,6
1,7,6,3,2,7,1,6,3,3,3,...,4,5,1,1,4,5,1,6,0,6
2,7,11,8,9,10,8,7,13,12,6,...,14,5,5,7,8,14,8,8,7,2
3,12,8,13,5,9,6,9,13,13,11,...,10,3,4,6,8,14,8,7,8,6
4,8,5,13,11,6,7,9,14,9,9,...,11,7,12,6,6,5,11,8,9,6


## 7. Yearly average sales of each cluster
Group by cluster label and take the mean of each week.

In [8]:
cluster_avg = df_w.groupby("label").mean()
cluster_avg

,W0,W1,W2,W3,W4,W5,W6,W7,W8,W9,...,W42,W43,W44,W45,W46,W47,W48,W49,W50,W51
label,,,,,,,,,,,,,,,,,,,,,
0,0.000000,0.375000,0.125000,0.250000,0.000000,0.000000,0.625000,0.125000,0.125000,0.500000,...,0.125000,0.250000,0.000000,0.875000,0.500000,0.125000,0.000000,0.500000,0.000000,0.125000
1,7.988889,8.477778,8.488889,8.477778,9.288889,9.100000,8.411111,8.822222,9.933333,9.366667,...,9.444444,9.000000,8.633333,8.566667,9.066667,9.811111,9.811111,11.722222,12.044444,14.488889
2,10.562500,10.427083,9.802083,10.385417,10.187500,11.031250,10.760417,10.635417,10.177083,9.416667,...,8.614583,8.083333,8.312500,8.531250,8.781250,9.229167,9.239583,10.114583,11.406250,12.666667
3,0.077381,0.130952,0.101190,0.077381,0.107143,0.107143,0.083333,0.101190,0.047619,0.119048,...,0.214286,0.184524,0.226190,0.184524,0.315476,0.154762,0.327381,0.303571,0.261905,0.464286
4,3.388235,2.988235,3.282353,3.729412,3.635294,3.505882,4.129412,4.105882,3.623529,3.752941,...,3.635294,3.847059,4.200000,3.729412,4.141176,4.717647,4.011765,4.458824,5.247059,7.141176
5,26.492188,28.148438,29.054688,29.218750,29.132812,28.445312,30.187500,28.609375,30.085938,29.445312,...,24.945312,24.781250,24.984375,25.257812,25.343750,24.773438,24.710938,23.695312,22.351562,18.296875
6,13.558140,13.317829,14.263566,15.511628,14.155039,13.868217,13.961240,14.612403,14.271318,15.046512,...,11.674419,11.930233,12.147287,12.713178,12.651163,12.046512,12.147287,12.124031,11.302326,10.829457
7,0.429907,0.373832,0.504673,0.448598,0.644860,0.495327,0.542056,0.420561,0.514019,0.485981,...,0.803738,0.831776,0.981308,1.130841,1.196262,1.028037,1.299065,1.467290,1.859813,2.467290


Visualise the average weekly sales trend of each cluster.

In [ ]:
cluster_avg.T.plot(figsize=(12, 6))
plt.xlabel("Week")
plt.ylabel("Average sales")
plt.title("Average weekly sales per cluster (k = 8)")
plt.legend(title="Cluster", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

## 8. Experiment with the number of clusters
To find a good `k`, we try k = 2 to 12 and compare the **inertia** (elbow method; lower is better, look for the bend) and the **silhouette score** (higher is better).

In [ ]:
ks = range(2, 13)
inertias, sil_scores = [], []

for k in ks:
    km = KMeans(n_clusters=k, random_state=8).fit(df_norm)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(df_norm, km.labels_))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(ks, inertias, marker="o")
ax[0].set_title("Elbow method")
ax[0].set_xlabel("Number of clusters (k)")
ax[0].set_ylabel("Inertia")

ax[1].plot(ks, sil_scores, marker="o", color="green")
ax[1].set_title("Silhouette score")
ax[1].set_xlabel("Number of clusters (k)")
ax[1].set_ylabel("Score")
plt.tight_layout()
plt.show()

In [ ]:
pd.DataFrame({"k": list(ks), "inertia": inertias, "silhouette": sil_scores})

## Conclusion
- With k = 8, k-means separates products into groups whose average weekly sales follow different trends (e.g. rising, falling, flat, seasonal peaks).
- The elbow and silhouette plots above show how the fit changes with `k`; choose the value where inertia stops dropping sharply and the silhouette score is high.
- *(Replace this note with your own observation of the optimal k after running the cells.)*